# NepNLP — Train a Nepali text classifier on Colab

Fine-tune a multilingual transformer for **Nepali sentiment or news classification**, and
compare it against a strong classical baseline. Runs on a free Colab **GPU**
(Runtime → Change runtime type → T4 GPU).

This notebook is self-contained: upload a `text,label` CSV, or run the tiny built-in sample.
For your real project, replace the sample with the dataset you collect
(see `docs/DATA_COLLECTION.md`).


## 1. Install dependencies


In [ ]:
!pip -q install 'transformers>=4.44' 'datasets>=2.20' evaluate accelerate scikit-learn sentencepiece


## 2. Get your data

Upload a UTF-8 CSV with two columns: `text` (Nepali) and `label`. If you skip the upload,
a small built-in sample is used so the notebook still runs.


In [ ]:
import pandas as pd
try:
    from google.colab import files
    print('Choose a text,label CSV (or press Cancel to use the built-in sample)...')
    up = files.upload()
    df = pd.read_csv(next(iter(up)))
except Exception as e:
    print('Using built-in sample:', e)
    data = [
        ('यो चलचित्र साह्रै राम्रो लाग्यो।','सकारात्मक'),
        ('उनको प्रस्तुति उत्कृष्ट थियो।','सकारात्मक'),
        ('खानाको स्वाद मीठो थियो।','सकारात्मक'),
        ('म यो उत्पादनबाट खुसी छु।','सकारात्मक'),
        ('सेवा एकदमै नराम्रो थियो।','नकारात्मक'),
        ('गुणस्तर खराब लाग्यो।','नकारात्मक'),
        ('म यो निर्णयबाट निराश भएँ।','नकारात्मक'),
        ('सेवा ढिलो र झन्झटिलो छ।','नकारात्मक'),
        ('बैठक दश बजे सुरु हुनेछ।','तटस्थ'),
        ('उनी काठमाडौंमा बस्छन्।','तटस्थ'),
        ('यो पुस्तकमा बाह्र अध्याय छन्।','तटस्थ'),
        ('कार्यालय पाँच बजे बन्द हुन्छ।','तटस्थ'),
    ]
    df = pd.DataFrame(data, columns=['text','label'])
df = df.dropna(subset=['text','label'])
print(df.shape)
df['label'].value_counts()


## 3. Nepali preprocessing

Minimal version of `ml-service/app/preprocessing.py` — NFC normalization, zero-width removal,
digit normalization. Keeping train/serve preprocessing identical avoids subtle accuracy loss.


In [ ]:
import re, unicodedata
ZW = re.compile('[\u200b\u200c\u200d\ufeff\u00ad]')
DEV2ASCII = {ord(d):a for d,a in zip('०१२३४५६७८९','0123456789')}
def clean(t):
    t = unicodedata.normalize('NFC', str(t))
    t = ZW.sub('', t)
    t = t.translate(DEV2ASCII)
    t = re.sub(r'https?://\S+',' ', t)
    t = re.sub(r'\s+',' ', t).strip()
    return t
df['text'] = df['text'].map(clean)
df.head()


## 4. Baseline — TF-IDF + Logistic Regression

Always report a baseline. 'My transformer beats a strong baseline by X points' is far more
credible than a lone accuracy number.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

Xtr, Xte, ytr, yte = train_test_split(df['text'], df['label'], test_size=0.25,
                                      random_state=42, stratify=df['label'])
base = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)),
                 ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
base.fit(Xtr, ytr)
print(classification_report(yte, base.predict(Xte), zero_division=0))


## 5. Fine-tune a transformer

Good Nepali-friendly base models: `xlm-roberta-base`, `google/muril-base-cased`,
`Sakonii/distilbert-base-nepali`, `NepBERTa/NepBERTa`.


In [ ]:
import numpy as np, evaluate
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

BASE_MODEL = 'xlm-roberta-base'  # try 'google/muril-base-cased' too
labels = sorted(df['label'].unique())
l2id = {l:i for i,l in enumerate(labels)}; id2l = {i:l for l,i in l2id.items()}

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
def tk(b): return tok(b['text'], truncation=True, max_length=256)
dtr = Dataset.from_dict({'text':list(Xtr),'label':[l2id[y] for y in ytr]}).map(tk, batched=True)
dte = Dataset.from_dict({'text':list(Xte),'label':[l2id[y] for y in yte]}).map(tk, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(labels), id2label=id2l, label2id=l2id)

acc = evaluate.load('accuracy'); f1 = evaluate.load('f1')
def metrics(p):
    pr = np.argmax(p.predictions, axis=-1)
    return {'accuracy':acc.compute(predictions=pr, references=p.label_ids)['accuracy'],
            'f1_macro':f1.compute(predictions=pr, references=p.label_ids, average='macro')['f1']}

args = TrainingArguments(output_dir='out', num_train_epochs=4, per_device_train_batch_size=16,
    per_device_eval_batch_size=16, learning_rate=2e-5, eval_strategy='epoch',
    save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1_macro',
    logging_steps=10, report_to='none', seed=42)
trainer = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=dte,
    tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()


## 6. Evaluate — per-class report + confusion matrix


In [ ]:
import matplotlib.pyplot as plt
pred = np.argmax(trainer.predict(dte).predictions, axis=-1)
gold = [l2id[y] for y in yte]
print(classification_report(gold, pred, target_names=labels, zero_division=0))
cm = confusion_matrix(gold, pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.title('Confusion matrix'); plt.tight_layout(); plt.show()


## 7. Save & (optionally) publish

Save the model so the NepNLP ML service can load it. Copy the folder into
`ml-service/models_store/<sentiment|news_classifier>/` and the app auto-upgrades to it.


In [ ]:
OUT = 'nepnlp-sentiment'  # or 'nepnlp-news'
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('saved to', OUT)

# Download the folder as a zip:
# !zip -r model.zip $OUT && from google.colab import files; files.download('model.zip')

# Or push to the Hugging Face Hub (needs `huggingface-cli login`):
# trainer.push_to_hub('your-username/nepnlp-sentiment')
# tok.push_to_hub('your-username/nepnlp-sentiment')
